In [16]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

In [17]:
games = pd.read_csv('games.csv')
clubs = pd.read_csv('clubs.csv')

In [18]:
games.head()

,game_id,competition_id,season,round,date,home_club_id,away_club_id,home_club_goals,away_club_goals,home_club_position,...,stadium,attendance,referee,url,home_club_formation,away_club_formation,home_club_name,away_club_name,aggregate,competition_type
0,1026846,FIWC,2009,Round of 16,2010-06-26,3449,3589,2,1,NaN,...,Nelson Mandela Bay Stadium,30597.0,Wolfgang Stark,https://www.transfermarkt.co.uk/spielbericht/i...,NaN,NaN,Uruguay,South Korea,2:1,national_team_competition
1,1026847,FIWC,2009,Round of 16,2010-06-27,3437,6303,3,1,NaN,...,FNB-Stadium,84377.0,Roberto Rosetti,https://www.transfermarkt.co.uk/spielbericht/i...,NaN,NaN,Argentina,Mexico,3:1,national_team_competition
2,1027001,FIWC,2009,Round of 16,2010-06-26,3505,3441,1,2,NaN,...,Royal Bafokeng Stadium,34976.0,Viktor Kassai,https://www.transfermarkt.co.uk/spielbericht/i...,NaN,NaN,United States,Ghana,1:2,national_team_competition
3,1027002,FIWC,2009,Round of 16,2010-06-27,3262,3299,4,1,NaN,...,Free State Stadium,40510.0,Jorge Larrionda,https://www.transfermarkt.co.uk/spielbericht/i...,NaN,NaN,Germany,England,4:1,national_team_competition
4,1027048,FIWC,2009,Round of 16,2010-06-28,3379,3503,2,1,NaN,...,Moses Mabhida Stadion,61962.0,Undiano Mallenco,https://www.transfermarkt.co.uk/spielbericht/i...,NaN,NaN,Netherlands,Slovakia,2:1,national_team_competition


In [19]:
clubs.head()

,club_id,club_code,name,domestic_competition_id,total_market_value,squad_size,average_age,foreigners_number,foreigners_percentage,national_team_players,stadium_name,stadium_seats,net_transfer_record,coach_name,last_season,filename,url
0,10,arminia-bielefeld,Arminia Bielefeld,L1,NaN,27,25.3,15,55.6,4,SchücoArena,26515,+€5.90m,NaN,2021,../data/raw/transfermarkt-scraper/2021/clubs.j...,https://www.transfermarkt.co.uk/arminia-bielef...
1,10004,paris-fc,Paris Football Club,FR1,NaN,31,28.5,17,54.8,8,Stade Jean Bouin,19904,€-72.30m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/paris-fc/start...
2,10010,esporte-clube-bahia,Esporte Clube Bahia,BRA1,NaN,32,26.2,6,18.8,3,Arena Fonte Nova,47364,+€8.14m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/esporte-clube-...
3,1003,leicester-city,Leicester City,GB1,NaN,29,25.9,17,58.6,9,King Power Stadium,32259,+€57.30m,Steve Cooper,2024,../data/raw/transfermarkt-scraper/2024/clubs.j...,https://www.transfermarkt.co.uk/leicester-city...
4,1005,us-lecce,Unione Sportiva Lecce,IT1,NaN,27,25.1,23,85.2,10,Ettore Giardiniero,31559,+€8.62m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/us-lecce/start...


In [20]:
def determine_result(row):
    if row['home_club_goals'] > row['away_club_goals']: return 1
    if row['home_club_goals'] == row['away_club_goals']: return 0
    return 2

In [21]:
df = games[['home_club_id', 'away_club_id', 'home_club_goals', 'away_club_goals']].dropna()
df['target'] = df.apply(determine_result, axis=1)

In [22]:
club_le = LabelEncoder()
all_ids = pd.concat([df['home_club_id'], df['away_club_id']]).unique()
club_le.fit(all_ids)

LabelEncoder()

In [23]:
df['home_idx'] = club_le.transform(df['home_club_id'])
df['away_idx'] = club_le.transform(df['away_club_id'])

In [24]:
# --- STEP 4: Training ---
X = df[['home_idx', 'away_idx']]
y = df['target']

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

RandomForestClassifier(random_state=42)

In [32]:
# '3' is a good balance between speed and file size
joblib.dump(model, 'football_model.pkl', compress=3)

['football_model.pkl']

In [35]:

joblib.dump(club_le, 'club-encoder.pkl')

['club-encoder.pkl']

In [29]:
name_map = dict(zip(clubs['club_id'], clubs['name']))
joblib.dump(name_map, 'name_map.pkl')

print("Success! Download football_model.pkl, club_encoder.pkl, and name_map.pkl from the sidebar.")

Success! Download football_model.pkl, club_encoder.pkl, and name_map.pkl from the sidebar.
